# 🏥 Advanced Clinical AI: Hospital 30-Day Readmission Risk Prediction
### Multi-Model Machine Learning Pipeline, Leakage-Free Feature Engineering & Clinical Decision Support
**Dataset**: UCI Diabetes 130-US Hospitals (1999–2008)  
**Target**: Early Readmission within 30 days (`readmitted < 30`)  
**Architecture**: Scikit-Learn Pipeline, Logistic Regression, Random Forest, XGBoost, LightGBM


## 1. Setup & Dependencies
Importing core scientific libraries, visualization frameworks, and scikit-learn pipeline modules.


In [ ]:
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
    brier_score_loss, log_loss
)
import joblib

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Environment initialized successfully.")


## 2. Data Ingestion & Initial Integrity Auditing
Loading the raw dataset `diabetic_data.csv`. We handle missing sentinel symbols (`?`) and inspect dimensions.


In [ ]:
data_path = Path('../diabetic_data.csv')
if not data_path.exists():
    data_path = Path('diabetic_data.csv')

df_raw = pd.read_csv(data_path, na_values=['?'], low_memory=False)
print(f"Dataset Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()


## 3. Missing Value Analysis & Clinical Auditing
Examining columns with severe missingness (`weight` > 96%, `payer_code` ~40%, `medical_specialty` ~49%).
Features with extreme sparsity or uninformative signals are systematically dropped.


In [ ]:
missing_summary = df_raw.isnull().sum()
missing_pct = (missing_summary / len(df_raw)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_summary, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)
print("Missing values by feature:")
print(missing_df)

# Visualizing top missing attributes
plt.figure(figsize=(9, 4))
missing_df['Percentage'].plot(kind='bar', color='#e74c3c')
plt.title("Missing Value Distribution (%)", fontsize=13, fontweight='bold')
plt.ylabel("Percent Missing (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Clinical Target Formulation: 30-Day Readmission
The original `readmitted` feature contains 3 classes:
- `<30`: Unplanned readmission within 30 days (**Primary clinical target of interest = 1**)
- `>30`: Readmission after 30 days (**Non-acute = 0**)
- `NO`: No readmission recorded (**Non-acute = 0**)

Early 30-day readmissions trigger CMS Hospital Readmissions Reduction Program (HRRP) penalties and represent high-risk transitions of care.


In [ ]:
print("Original readmission breakdown:")
print(df_raw['readmitted'].value_counts())

# Binary mapping: 1 if <30, else 0
df = df_raw.copy()
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

pos_count = df['readmitted_binary'].sum()
pos_rate = df['readmitted_binary'].mean() * 100
print(f"\nTarget Distribution:")
print(f"Negative class (0): {len(df) - pos_count:,} ({100 - pos_rate:.2f}%)")
print(f"Positive class (1): {pos_count:,} ({pos_rate:.2f}%)")
print(f"Imbalance ratio: 1 : {(len(df) - pos_count) / pos_count:.1f}")


## 5. Cohort Filtering & Deduplication
To prevent patient-level data leakage:
1. Multiple hospitalizations for the same patient (`patient_nbr`) can create spurious correlation. We retain the first recorded encounter per patient.
2. We exclude deceased patients or those transferred to hospice (`discharge_disposition_id` in [11, 13, 14, 19, 20, 21]), as they cannot be readmitted.


In [ ]:
# Filter out deceased / hospice patients
terminal_dispositions = [11, 13, 14, 19, 20, 21]
df_filtered = df[~df['discharge_disposition_id'].isin(terminal_dispositions)].copy()
print(f"Cohort after excluding terminal encounters: {len(df_filtered):,} (removed {len(df) - len(df_filtered):,})")

# Deduplicate to first encounter per patient
df_unique = df_filtered.drop_duplicates(subset=['patient_nbr'], keep='first').copy()
print(f"Cohort after patient deduplication: {len(df_unique):,} unique patients")
print(f"Unique patient 30-day readmission rate: {df_unique['readmitted_binary'].mean()*100:.2f}%")


## 6. Advanced Clinical Feature Engineering
We construct domain-specific predictors:
1. **ICD-9 Diagnostic Category Mapping** (`diag_1_category`): Circulatory, Respiratory, Digestive, Diabetes, Injury, Musculoskeletal, Genitourinary, Neoplasms, Other.
2. **Total Prior Encounters** (`total_past_admissions`): Sum of outpatient, emergency, and inpatient visits.
3. **Admission Risk Tier** (`admission_risk_level`): Stratified utilization index.
4. **Polypharmacy Index** (`polypharmacy`): >20 concurrent medications administered.
5. **Medication Adjustments** (`n_med_changes`): Count of anti-diabetic medications modified or titrated during stay.
6. **Age Numeric Midpoint** (`age_numeric`): Converting age brackets `[60-70)` to numeric midpoints.


In [ ]:
# 1. ICD-9 Diagnostic category mapper
def categorize_icd9(val):
    try:
        val_str = str(val).strip()
        if val_str.startswith('V') or val_str.startswith('E'):
            return 'Other'
        v = float(val_str)
        if (390 <= v <= 459) or v == 785:
            return 'Circulatory'
        elif (460 <= v <= 519) or v == 786:
            return 'Respiratory'
        elif (520 <= v <= 579) or v == 787:
            return 'Digestive'
        elif 250 <= v < 251:
            return 'Diabetes'
        elif 800 <= v <= 999:
            return 'Injury'
        elif 710 <= v <= 739:
            return 'Musculoskeletal'
        elif (580 <= v <= 629) or v == 788:
            return 'Genitourinary'
        elif 140 <= v <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except:
        return 'Other'

df_unique['diag_1_category'] = df_unique['diag_1'].apply(categorize_icd9)

# 2. Total prior admissions
df_unique['total_past_admissions'] = (
    df_unique['number_outpatient'] + df_unique['number_emergency'] + df_unique['number_inpatient']
)

# 3. Admission risk level
def stratify_risk(visits):
    if visits == 0: return 'None'
    elif 1 <= visits <= 3: return 'Low'
    elif 4 <= visits <= 10: return 'Medium'
    else: return 'High'

df_unique['admission_risk_level'] = df_unique['total_past_admissions'].apply(stratify_risk)

# 4. Polypharmacy (>20 medications)
df_unique['polypharmacy'] = (df_unique['num_medications'] > 20).astype(int)

# 5. Age numeric midpoint
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df_unique['age_numeric'] = df_unique['age'].map(age_map).fillna(65).astype(int)

# 6. Binary indicators
df_unique['diabetesMed_binary'] = (df_unique['diabetesMed'] == 'Yes').astype(int)
df_unique['change_binary'] = (df_unique['change'] == 'Ch').astype(int)
df_unique['insulin_change'] = df_unique['insulin'].isin(['Up', 'Down']).astype(int)
df_unique['A1C_abnormal'] = df_unique['A1Cresult'].isin(['>7', '>8']).astype(int)
df_unique['max_glu_abnormal'] = df_unique['max_glu_serum'].isin(['>200', '>300']).astype(int)

med_cols = ['metformin', 'repaglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin']
df_unique['n_med_changes'] = df_unique[[c for c in med_cols if c in df_unique.columns]].apply(
    lambda row: sum(val in ['Up', 'Down'] for val in row), axis=1
)

# Clean demographic categories
df_unique['gender'] = df_unique['gender'].replace('Unknown/Invalid', 'Female')
df_unique['race'] = df_unique['race'].fillna('Unknown')

print("Engineered features successfully created.")


## 7. Exploratory Data Analysis & Clinical Insights
We inspect the relationship between patient features and 30-day readmission risk.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Readmission by Prior Inpatient Admissions
sns.barplot(data=df_unique, x='number_inpatient', y='readmitted_binary', ax=axes[0, 0], ci=None, color='#3498db')
axes[0, 0].set_title('Readmission Rate by Prior Inpatient Admissions', fontweight='bold')
axes[0, 0].set_ylabel('Readmission Rate')
axes[0, 0].set_xlim(-0.5, 6.5)

# 2. Readmission by Diagnosis Category
diag_order = df_unique.groupby('diag_1_category')['readmitted_binary'].mean().sort_values(ascending=False).index
sns.barplot(data=df_unique, x='readmitted_binary', y='diag_1_category', order=diag_order, ax=axes[0, 1], ci=None, palette='Blues_r')
axes[0, 1].set_title('Readmission Rate by Primary Diagnosis', fontweight='bold')
axes[0, 1].set_xlabel('Readmission Rate')

# 3. Readmission by Age Bracket
age_order = sorted(df_unique['age'].unique(), key=lambda x: int(x.split('-')[0].replace('[', '')))
sns.barplot(data=df_unique, x='age', y='readmitted_binary', order=age_order, ax=axes[1, 0], ci=None, color='#2ecc71')
axes[1, 0].set_title('Readmission Rate across Age Cohorts', fontweight='bold')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)
axes[1, 0].set_ylabel('Readmission Rate')

# 4. Length of Stay vs Readmission
sns.barplot(data=df_unique, x='time_in_hospital', y='readmitted_binary', ax=axes[1, 1], ci=None, color='#e67e22')
axes[1, 1].set_title('Readmission Rate by Days in Hospital', fontweight='bold')
axes[1, 1].set_ylabel('Readmission Rate')

plt.tight_layout()
plt.show()


## 8. Leakage-Free Preprocessing Pipeline & Stratified Splitting
We split into Train (80%) and Test (20%) using stratification on the binary target.
All scaling and one-hot encoding are strictly fit within training folds only.


In [ ]:
feature_cols_numeric = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'age_numeric',
    'total_past_admissions', 'n_med_changes', 'polypharmacy',
    'diabetesMed_binary', 'change_binary', 'insulin_change',
    'A1C_abnormal', 'max_glu_abnormal', 'admission_type_id'
]

feature_cols_categorical = [
    'gender', 'race', 'diag_1_category', 'admission_risk_level'
]

all_features = feature_cols_numeric + feature_cols_categorical
X = df_unique[all_features].copy()
y = df_unique['readmitted_binary'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples | Readmission rate: {y_train.mean()*100:.2f}%")
print(f"Testing set:  {X_test.shape[0]:,} samples | Readmission rate: {y_test.mean()*100:.2f}%")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), feature_cols_numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), feature_cols_categorical)
    ]
)


## 9. Multi-Model Benchmark & Cross-Validation
We compare multiple candidate algorithms:
1. **Logistic Regression** (L2 penalty, class_weight='balanced')
2. **Random Forest** (n_estimators=100, class_weight='balanced')
3. **Histogram Gradient Boosting** (native class weights)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(class_weight='balanced', max_iter=100, random_state=42)
}

benchmark_results = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred_proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    
    benchmark_results[name] = {
        "ROC-AUC": round(roc_auc, 4),
        "PR-AUC": round(pr_auc, 4),
        "F1-Score": round(f1, 4),
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "Pipeline": pipe
    }

results_df = pd.DataFrame(benchmark_results).T.drop(columns=['Pipeline'])
print("\n=== Model Comparison Benchmark ===")
print(results_df)


## 10. Clinical Performance Curves (ROC & Precision-Recall)
Visualizing discriminative performance on the held-out test cohort.


In [ ]:
plt.figure(figsize=(12, 5))

# ROC Curve
plt.subplot(1, 2, 1)
for name, res in benchmark_results.items():
    pipe = res["Pipeline"]
    y_scores = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['ROC-AUC']})")
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title("Receiver Operating Characteristic (ROC)", fontweight='bold')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.legend(loc='lower right')

# Precision-Recall Curve
plt.subplot(1, 2, 2)
for name, res in benchmark_results.items():
    pipe = res["Pipeline"]
    y_scores = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_scores)
    plt.plot(rec, prec, label=f"{name} (PR-AUC = {res['PR-AUC']})")
plt.axhline(y=y_test.mean(), color='r', linestyle='--', label=f'Baseline ({y_test.mean()*100:.1f}%)')
plt.title("Precision-Recall Curve", fontweight='bold')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()


## 11. Model Interpretability & Feature Coefficients
Extracting odds ratios and standardized linear coefficients from the top performing clinical model.


In [ ]:
best_pipe = benchmark_results["Logistic Regression"]["Pipeline"]
best_clf = best_pipe.named_steps['classifier']
ohe = best_pipe.named_steps['preprocessor'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(feature_cols_categorical))
feat_names = feature_cols_numeric + cat_names

coefs = best_clf.coef_[0]
importance_df = pd.DataFrame({'Feature': feat_names, 'Coefficient': coefs})
importance_df['Abs_Weight'] = importance_df['Coefficient'].abs()
importance_df = importance_df.sort_values('Abs_Weight', ascending=False)

print("Top 15 Most Influential Clinical Features:")
print(importance_df.head(15)[['Feature', 'Coefficient']])

# Plot
plt.figure(figsize=(10, 6))
top15 = importance_df.head(15).sort_values('Coefficient')
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in top15['Coefficient']]
plt.barh(top15['Feature'], top15['Coefficient'], color=colors)
plt.axvline(0, color='black', linestyle='--', alpha=0.7)
plt.title("Top Clinical Feature Weights (Logistic Regression)", fontsize=13, fontweight='bold')
plt.xlabel("Coefficient Weight (Positive = Increases Readmission Risk)")
plt.tight_layout()
plt.show()


## 12. Model Serialization for Production FastAPI Serving
Saving the trained Scikit-Learn pipeline and associated metadata for low-latency web application serving.


In [ ]:
out_dir = Path('../backend/artifacts')
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipe, out_dir / 'model_pipeline.joblib')
print(f"Pipeline saved to {out_dir / 'model_pipeline.joblib'}")
print("Notebook execution complete.")
